In [ ]:
# Cell 153 (06_rq3_setup.ipynb, cell 1): superseded notes + shared/src/checks.py with tests. FRESH KERNEL. Exclusive create.
CHECKS_SRC = '''"""Convergence checks for sklearn estimators with n_iter_ (e.g. LogisticRegression).
assert_converged: n_iter_ < max_iter and no ConvergenceWarning. fit_converged: fit under a warning catcher, then assert."""
import warnings
import numpy as np
from sklearn.exceptions import ConvergenceWarning


def assert_converged(model, max_iter=None, caught=None, name="model"):
    mi = model.max_iter if max_iter is None else max_iter
    ni = int(np.max(model.n_iter_))
    conv_w = [w for w in (caught or []) if issubclass(w.category, ConvergenceWarning)]
    assert ni < mi, "%s did not converge: n_iter_ %d reached max_iter %d" % (name, ni, mi)
    assert not conv_w, "%s raised %d ConvergenceWarning(s)" % (name, len(conv_w))
    return ni


def fit_converged(model, X, y, name="model", **fit_kw):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(X, y, **fit_kw)
    assert_converged(model, model.max_iter, caught, name)
    return model
'''

def _c153():
    import os, sys, json, hashlib, importlib.util, warnings
    import numpy as np
    from sklearn.datasets import make_classification
    from sklearn.linear_model import LogisticRegression
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    NOTES = "selection/tables/sel_superseded_notes.json"
    ss.check_path(os.path.join(R, NOTES))
    cp = os.path.join(R, "shared/src/checks.py")
    assert not os.path.exists(cp), "checks.py already exists: STOP"

    ns = {}; exec(compile(CHECKS_SRC, "checks_candidate", "exec"), ns)
    X, y = make_classification(n_samples=3000, n_features=10, random_state=0)
    ok = ns["fit_converged"](LogisticRegression(solver="lbfgs", max_iter=5000), X, y, name="converging test")
    print("converging fit: passed, n_iter_ %d" % int(np.max(ok.n_iter_)))
    rejected = False
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ns["fit_converged"](LogisticRegression(solver="lbfgs", max_iter=2), X, y, name="capped test")
    except AssertionError as e:
        rejected = True; print("capped fit: correctly rejected ->", str(e)[:90])
    assert rejected, "capped fit was not rejected: STOP"

    with open(cp, "x", encoding="utf-8") as f: f.write(CHECKS_SRC)
    assert hashlib.sha256(open(cp, "rb").read()).hexdigest() == hashlib.sha256(CHECKS_SRC.encode()).hexdigest()
    os.chmod(cp, 0o444)
    spec = importlib.util.spec_from_file_location("checks_saved", cp); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    m.fit_converged(LogisticRegression(solver="lbfgs", max_iter=5000), X, y)
    print("saved shared/src/checks.py sha12 %s (re-imported from disk, passes)" % hashlib.sha256(CHECKS_SRC.encode()).hexdigest()[:12])

    notes = dict(
        purpose="Marks results superseded by converged re-fits. Old files are kept on disk unchanged.",
        superseded=[
            dict(files=["selection/tables/sel_precheck_table.csv", "selection/tables/sel_precheck_bootstrap.csv",
                        "selection/tables/sel_precheck_notes.json", "selection/predictions/sel_precheck_predictions.pkl"],
                 reason="Phase 1 LR used saga, C=100, max_iter=1000, which hit max_iter (not converged): Spec A AUC 0.6197, Spec B 0.6149, A-B p=0.558",
                 superseded_by=["selection/tables/sel_precheck_converged_table.csv", "selection/tables/sel_precheck_converged_bootstrap.csv", "selection/tables/sel_precheck_converged_notes.json"],
                 corrected="lbfgs converged: Spec A 0.6903, Spec B 0.6866, A-B +0.0038 (95% CI +0.0007, +0.0070), p=0.025"),
            dict(files=["selection/tables/sel_baselines_table.csv (row model=LR_additive_AgeDebt (diagnostic))",
                        "selection/tables/sel_baselines_bootstrap.csv (row comparison=BN - LR_additive (diagnostic))"],
                 reason="additive Age + Debt Ratio LR was fitted with saga and hit max_iter=1000: AUC 0.6228, BN - additive +0.0399",
                 superseded_by=["selection/tables/sel_additive_lbfgs_diagnostic.csv"],
                 corrected="lbfgs converged: AUC 0.6582, BN - additive +0.0045 (95% CI +0.0011, +0.0078), p=0.008")],
        not_superseded="Locked full-variable LR (saga, n_iter_ 211) is confirmed effectively converged: see sel_lockedLR_convergence_check.csv/.json")
    data = json.dumps(notes, indent=1).encode()
    p = ss.check_path(os.path.join(R, NOTES))
    with open(p, "xb") as f: f.write(data)
    assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
    print("saved sel_superseded_notes.json sha12 %s" % hashlib.sha256(data).hexdigest()[:12])
_c153()

In [ ]:
# Cell 154 (06_rq3_setup.ipynb): Part 0, ablation LR convergence (read-only on full_variable/). Run after Cell 153, no restart.
def _c154():
    import os, sys, json, pickle, hashlib, importlib.util, warnings
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    spec = importlib.util.spec_from_file_location("checks", R + "/shared/src/checks.py"); ck = importlib.util.module_from_spec(spec); spec.loader.exec_module(ck)
    O_C, O_J = "selection/tables/sel_ablationLR_convergence_check.csv", "selection/tables/sel_ablationLR_convergence_check.json"
    for rel in (O_C, O_J): ss.check_path(os.path.join(R, rel))
    L = lambda p: pickle.load(open(p, "rb"))
    Xtr, Xte = L(R + "/Data/processed/X_train_df.pkl"), L(R + "/Data/processed/X_test_df.pkl")
    sp = L(R + "/Data/processed/split_indices.pkl"); y_tr, y_te = np.asarray(sp["y_train"]).astype(int), np.asarray(sp["y_test"]).astype(int)
    abl = L(R + "/full_variable/predictions/ablation_predictions.pkl")
    MB = ['30-59 Days Past Due (bin)', '60-89 Days Past Due (bin)', '90+ Days Late (bin)', 'Credit Utilisation Rate (bin)']
    cols_for = lambda vs: [c for c in Xtr.columns if any(c.startswith(v + "_") for v in vs)]
    four = cols_for(MB); noid = [c for c in Xtr.columns if c not in cols_for(['Monthly Income (bin)', 'Number of Dependents (bin)'])]
    print("columns: 4-blanket %d (expect 13) | no-Income/Dependents %d (expect 28)" % (len(four), len(noid)))
    assert len(four) == 13 and len(noid) == 28
    rows, stop = [], False
    for name, cols, key, exp in [("LR_4var", four, "LR_4var", 0.8508), ("LR_noID", noid, "LR_noID", None)]:
        m = ck.fit_converged(LogisticRegression(C=100, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000), Xtr[cols].values, y_tr, name=name)
        p = m.predict_proba(Xte[cols].values)[:, 1]; pl = np.asarray(abl[key], float)
        a_new, a_old = roc_auc_score(y_te, p), roc_auc_score(y_te, pl)
        d = a_new - a_old; mx = float(np.max(np.abs(p - pl)))
        print("%-8s lbfgs n_iter_ %d, no warnings | AUC converged %.4f | locked (saved predictions) %.4f%s | diff %+.5f | max |prob diff| %.3e" % (
            name, int(np.max(m.n_iter_)), a_new, a_old, "" if exp is None else " (expected %.4f)" % exp, d, mx))
        if exp is not None: assert round(a_old, 4) == exp, "saved locked AUC differs from the expected value: STOP"
        stop |= abs(d) > 0.002
        rows.append(dict(model=name, n_columns=len(cols), solver="lbfgs", C=100, n_iter=int(np.max(m.n_iter_)), converged_auc=a_new, locked_auc=a_old,
                         auc_diff=d, max_abs_prob_diff=mx, locked_solver="saga", locked_max_iter=1000, locked_n_iter="unknown (object not saved)"))
    print("STOP RULE (any |AUC diff| > 0.002):", "TRIGGERED, nothing saved" if stop else "not triggered")
    if stop: return None
    t = pd.DataFrame(rows)
    def wr(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        return hashlib.sha256(data).hexdigest()[:12]
    print("saved:", wr(O_C, t.to_csv(index=False).encode()), wr(O_J, json.dumps(dict(rows=rows, note="read-only on full_variable/; converged via checks.fit_converged"), indent=1, default=float).encode()))
_c154()

In [ ]:
# Cell 155 (06_rq3_setup.ipynb): Part A step 1, READ-ONLY. Prints safe_save.py in full. Writes NOTHING. No restart.
def _c155():
    import os
    R = os.environ["CREDIT_BN_ROOT"]
    p = R + "/shared/src/safe_save.py"
    print("mode: %o | size %d bytes\n" % (os.stat(p).st_mode & 0o777, os.path.getsize(p)))
    print(open(p).read())
    print("\nmonitoring_rq3 exists:", os.path.exists(R + "/monitoring_rq3"))
_c155()

In [ ]:
# Cell 156 (06_rq3_setup.ipynb): Part A step 2. safe_save mon_ extension: test candidate in memory, then back up, replace, re-test from disk. No restart.
NEW_SS = '''"""safe_save: guarded saving for the SELECTION (sel_) and MONITORING RQ3 (mon_) pipelines.

Rules (all enforced):
  1. File name must start with a known prefix: 'sel_' or 'mon_' (or the prefix passed explicitly).
  2. Resolved path (case-insensitive, symlinks followed) must NOT be inside full_variable/ or archive/.
  3. 'sel_' files must be inside selection/ or Data/processed/. 'mon_' files must be inside monitoring_rq3/ ONLY.
  4. An existing path is refused unless overwrite=True.
Writes go to a temp file in the same folder, then os.replace (atomic).
"""
import os, json, pickle, hashlib

ROOT = os.environ["CREDIT_BN_ROOT"]
FORBIDDEN = ["full_variable", "archive"]
ALLOWED = ["selection", "data/processed"]          # allowed folders for the default 'sel_' prefix (kept for compatibility)
PREFIX = "sel_"
RULES = {"sel_": ["selection", "data/processed"], "mon_": ["monitoring_rq3"]}


class SaveRefused(Exception):
    pass


def _norm(p):
    return os.path.normcase(os.path.realpath(p)).lower()


def _inside(path, folder, root):
    f = _norm(os.path.join(root, folder))
    return path == f or path.startswith(f + os.sep)


def check_path(path, overwrite=False, root=ROOT, prefix=None):
    """Return the resolved path, or raise SaveRefused."""
    rp = _norm(path)
    name = os.path.basename(rp)
    if prefix is None:
        pf = next((k for k in RULES if name.startswith(k)), None)
        if pf is None:
            raise SaveRefused(f"file name must start with one of {list(RULES)}: {os.path.basename(path)}")
    else:
        pf = prefix.lower()
        if not name.startswith(pf):
            raise SaveRefused(f"file name must start with '{prefix}': {os.path.basename(path)}")
        if pf not in RULES:
            raise SaveRefused(f"unknown prefix '{prefix}' (known: {list(RULES)})")
    for f in FORBIDDEN:
        if _inside(rp, f, root):
            raise SaveRefused(f"path is inside protected folder '{f}/': {path}")
    allowed = RULES[pf]
    if not any(_inside(rp, a, root) for a in allowed):
        raise SaveRefused(f"'{pf}' files are only allowed inside {allowed}: {path}")
    if os.path.lexists(path) and not overwrite:
        raise SaveRefused(f"path already exists (use overwrite=True to replace): {path}")
    return os.path.realpath(path)


def _sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def safe_save(obj, path, kind=None, overwrite=False, root=ROOT, prefix=None, **kw):
    """Save obj to path. kind: pkl | json | csv | txt | png (inferred from extension if None).
    Returns (resolved_path, sha256)."""
    real = check_path(path, overwrite=overwrite, root=root, prefix=prefix)
    kind = kind or os.path.splitext(path)[1].lstrip(".").lower()
    os.makedirs(os.path.dirname(real), exist_ok=True)
    tmp = real + ".tmp_safe_save"
    try:
        if kind in ("pkl", "pickle"):
            with open(tmp, "wb") as f:
                pickle.dump(obj, f, protocol=4)
        elif kind == "json":
            with open(tmp, "w") as f:
                json.dump(obj, f, indent=2, default=str)
        elif kind == "csv":
            obj.to_csv(tmp, **kw)
        elif kind in ("txt", "md"):
            with open(tmp, "w") as f:
                f.write(obj)
        elif kind == "png":
            obj.savefig(tmp, format="png", **({"dpi": 200, "bbox_inches": "tight"} | kw))
        else:
            raise SaveRefused(f"unsupported kind: {kind}")
        if os.path.lexists(real) and not overwrite:   # re-check just before the swap
            raise SaveRefused(f"path appeared during save: {path}")
        os.replace(tmp, real)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
    return real, _sha256(real)
'''

def _c156():
    import os, sys, hashlib, importlib.util
    R = os.environ["CREDIT_BN_ROOT"]
    P = R + "/shared/src/safe_save.py"; BAK = R + "/shared/src/safe_save_before_mon.py.bak"
    assert not os.path.exists(BAK), "backup already exists: STOP"

    def run_tests(mod, label):
        S = mod.SaveRefused
        def refused(fn):
            try: fn(); return False
            except S: return True
        def allowed(fn):
            try: fn(); return True
            except S: return False
        c = mod.check_path
        T = [
            ("ORIG 1  existing path refused",            refused(lambda: c(R + "/selection/tables/sel_bn_metrics.json"))),
            ("ORIG 2a full_variable/ refused",           refused(lambda: c(R + "/full_variable/tables/sel_x.csv"))),
            ("ORIG 2b archive/ refused",                 refused(lambda: c(R + "/archive/sel_x.csv"))),
            ("ORIG 2c case-insensitive path refused",    refused(lambda: c(R + "/Full_Variable/tables/sel_x.csv"))),
            ("ORIG 3  missing prefix refused",           refused(lambda: c(R + "/selection/tables/notprefixed.csv"))),
            ("sel_ in selection/ allowed",               allowed(lambda: c(R + "/selection/tables/sel_zz_test.csv"))),
            ("sel_ in Data/processed allowed",           allowed(lambda: c(R + "/Data/processed/sel_zz_test.pkl"))),
            ("NEW 1   mon_ in monitoring_rq3/ allowed",  allowed(lambda: c(R + "/monitoring_rq3/mon_zz_test.csv"))),
            ("NEW 2   mon_ in selection/ refused",       refused(lambda: c(R + "/selection/tables/mon_zz_test.csv"))),
            ("NEW 3   sel_ in monitoring_rq3/ refused",  refused(lambda: c(R + "/monitoring_rq3/sel_zz_test.csv"))),
            ("extra   mon_ in Data/processed refused",   refused(lambda: c(R + "/Data/processed/mon_zz_test.pkl"))),
            ("extra   mon_ in full_variable refused",    refused(lambda: c(R + "/full_variable/mon_zz_test.csv"))),
            ("extra   explicit prefix='sel_' in monitoring_rq3 refused", refused(lambda: c(R + "/monitoring_rq3/sel_zz_test.csv", prefix="sel_"))),
            ("extra   explicit prefix='mon_' on a sel_ name refused",    refused(lambda: c(R + "/selection/tables/sel_zz_test.csv", prefix="mon_"))),
            ("extra   unknown prefix refused",           refused(lambda: c(R + "/selection/tables/abc_zz_test.csv", prefix="abc_"))),
        ]
        print("--- %s ---" % label)
        for n, ok in T: print("  %-58s %s" % (n, "PASS" if ok else "FAIL"))
        return all(ok for _, ok in T)

    ns = {"__name__": "safe_save_candidate"}; exec(compile(NEW_SS, "safe_save_candidate", "exec"), ns)
    cand = type("M", (), ns)
    assert run_tests(cand, "candidate (in memory)"), "candidate failed a test: nothing written. STOP"

    old = open(P, "rb").read()
    with open(BAK, "xb") as f: f.write(old)
    assert hashlib.sha256(open(BAK, "rb").read()).hexdigest() == hashlib.sha256(old).hexdigest()
    tmp = P + ".tmp_new"
    with open(tmp, "x", encoding="utf-8") as f: f.write(NEW_SS)
    os.replace(tmp, P)
    os.makedirs(R + "/monitoring_rq3", exist_ok=True)
    print("\nbackup: shared/src/safe_save_before_mon.py.bak sha12 %s | new safe_save.py sha12 %s" % (
        hashlib.sha256(old).hexdigest()[:12], hashlib.sha256(open(P, "rb").read()).hexdigest()[:12]))

    sys.modules.pop("safe_save", None)
    spec = importlib.util.spec_from_file_location("safe_save", P); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    sys.modules["safe_save"] = m
    ok = run_tests(m, "saved file, re-imported from disk")
    print("\nmonitoring_rq3/ exists:", os.path.isdir(R + "/monitoring_rq3"), "| ALL PASS:", ok)
    if not ok: print("FAILED after write: restore with the .bak and report")
_c156()

In [ ]:
# Cell 157 (06_rq3_setup.ipynb): Part B step 1, READ-ONLY. Imputer behaviour test + pilot-variable pattern counts. Writes NOTHING. No restart.
def _c157():
    import os, sys, json, pickle, warnings
    import numpy as np, pandas as pd
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    from sklearn.linear_model import BayesianRidge, LinearRegression
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    L = lambda p: pickle.load(open(p, "rb")); PROC = R + "/Data/processed/"
    warnings.filterwarnings("ignore")

    # 1. toy: fitted on complete data, then a column is masked at transform time
    rng = np.random.RandomState(0); n = 4000
    X = rng.normal(size=(n, 4)); X[:, 1] = 0.8 * X[:, 0] + 0.3 * rng.normal(size=n) + 5.0
    imp = IterativeImputer(estimator=BayesianRidge(), random_state=42).fit(X)
    print("toy: fitted on complete data. imputer.initial_imputer_ strategy:", imp.initial_imputer_.strategy,
          "| n_iter_", imp.n_iter_, "| imputation_sequence_ length", len(imp.imputation_sequence_))
    Xm = X.copy(); mask = rng.rand(n) < 0.3; Xm[mask, 1] = np.nan
    out = imp.transform(Xm)[mask, 1]
    mean_fill = X[:, 1].mean(); med_fill = np.median(X[:, 1])
    reg = LinearRegression().fit(np.delete(X, 1, 1), X[:, 1]).predict(np.delete(Xm[mask], 1, 1))
    print("  imputed values: distinct %d | min %.3f max %.3f" % (len(np.unique(np.round(out, 8))), out.min(), out.max()))
    print("  max |imputed - column mean| %.3e | max |imputed - median| %.3e | max |imputed - linear regression on others| %.3e" % (
        np.max(np.abs(out - mean_fill)), np.max(np.abs(out - med_fill)), np.max(np.abs(out - reg))))
    print("  -> if the first two are large and the third small, the imputer regresses (not the initial strategy only)")

    # 2. real Spec A design
    nb = json.load(open(R + "/selection/notebooks/00_precheck.ipynb")); code = "".join(nb["cells"][2]["source"])
    sp = L(PROC + "split_indices.pkl"); g = {"os": os, "np": np, "pd": pd, "L": L, "PROC": PROC, "ROOT": R}
    g["train_idx"], g["test_idx"] = np.asarray(sp["train_idx"]), np.asarray(sp["test_idx"])
    g["y_train"], g["y_test"] = np.asarray(sp["y_train"]), np.asarray(sp["y_test"])
    exec(compile(code[:code.index("# ---- validation")], "bd", "exec"), g)
    dfc = g["dfc"]; IO = g["IMPUTE_ORDER"]
    print("\nlocked IMPUTE_ORDER:", IO)
    print("DESIGN_ORDER:", g["DESIGN_ORDER"])
    tr = dfc.loc[g["train_idx"], IO].copy(); tr["Monthly Income"] = np.log1p(tr["Monthly Income"])
    print("train NaNs per column:", {c: int(tr[c].isna().sum()) for c in IO if tr[c].isna().any()})
    full_imp = IterativeImputer(random_state=42).fit(tr)
    print("locked-style imputer (BayesianRidge default) fitted on train WITH NaNs in Income/Dependents: n_iter_", full_imp.n_iter_,
          "| initial strategy", full_imp.initial_imputer_.strategy)
    cc = tr.dropna(); print("complete training rows: %d" % len(cc))
    imp_cc = IterativeImputer(estimator=BayesianRidge(), random_state=42).fit(cc)
    te = dfc.loc[g["test_idx"], IO].copy(); te["Monthly Income"] = np.log1p(te["Monthly Income"])
    tcc = te.dropna().head(2000).copy(); col = "Debt Ratio"
    tm = tcc.copy(); tm.loc[tm.index[::3], col] = np.nan; mk = tm[col].isna().to_numpy()
    o = imp_cc.transform(tm)[:, IO.index(col)][mk]
    print("real columns, imputer fitted on complete training rows, masked '%s' at transform: distinct imputed %d | max |imputed - train mean| %.3e | max |imputed - train median| %.3e | sd of imputed %.4f" % (
        col, len(np.unique(np.round(o, 8))), np.max(np.abs(o - cc[col].mean())), np.max(np.abs(o - cc[col].median())), o.std()))

    # 3. evidence patterns per candidate masked variable
    from pgmpy.models import DiscreteBayesianNetwork
    sn = json.load(open(PROC + "sel_state_names.json")); raw = pd.read_pickle(PROC + "sel_test.pkl")
    te_s = pd.DataFrame({c: (raw[c] if set(raw[c].dropna().unique()) <= set(sn[c]) else raw[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw.columns})
    ev_s = [c for c in te_s.columns if c != "Default"]
    print("\nSELECTION evidence columns:", ev_s)
    for v in ["Age (bin)", "Debt Ratio (bin)"]:
        cols = [c for c in ev_s if c != v]; print("  mask %-18s unique evidence patterns: %d" % (v, len(te_s[cols].astype(str).drop_duplicates())))
    dte = L(PROC + "df_test.pkl")
    ev_m = [c for c in dte.columns if c in ('Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)', '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)', 'M_Income', 'M_Dependents')]
    print("MONITORING evidence columns:", ev_m)
    for v in ['Credit Utilisation Rate (bin)', '30-59 Days Past Due (bin)', '90+ Days Late (bin)', '60-89 Days Past Due (bin)']:
        cols = [c for c in ev_m if c != v]; print("  mask %-30s unique evidence patterns: %d" % (v, len(dte[cols].astype(str).drop_duplicates())))

    # 4. objects
    for f in ("full_variable/models/imputer.pkl", "full_variable/models/encoder.pkl", "selection/models/sel_lr_grid.pkl", "selection/models/sel_rf_grid.pkl", "full_variable/models/lr_final.pkl", "full_variable/models/rf_final_restricted.pkl"):
        try:
            o = L(R + "/" + f); print("loads: %-48s %s" % (f, type(o).__name__))
        except Exception as e:
            print("FAILED:", f, type(e).__name__, str(e)[:80])
_c157()

In [ ]:
# Cell 158 (06_rq3_setup.ipynb): Part B step 2. shared/src/masking.py (make_mask + conditioning rule), tested in memory, then exclusive create. No restart.
MASK_SRC = '''"""Test-set masking for RQ3 (paired across models; one mask per contribution/variable/level/mechanism/seed).
make_mask(n_rows, level, mechanism, seed, cond_codes=None) -> (mask, realised_fraction); True = masked.
'mcar': each row masked independently with p = level.
'dependent': p_i = min(1, s * w_i / mean(w)), w_i = cond_codes_i + 1 (cond_codes in 0..4); one uniform u_i per row from
RandomState(seed); s is bisected so the REALISED masked fraction of (u < p) is within 0.005 of level (in practice ~1/n).
Conditioning variable rule (record in notes): Debt Ratio bin for every masked variable, EXCEPT Age bin when the masked
variable is Debt Ratio (selection)."""
import os
assert os.environ.get("PYTHONHASHSEED") == "0", "PYTHONHASHSEED != 0: wrong kernel"
import numpy as np

SEEDS = list(range(42, 52))          # 10 seeds
SEEDS_FALLBACK = list(range(42, 47)) # 5-seed fallback
LEVELS_TOL = 0.005


def cond_variable(masked_var):
    """Name of the always-observed conditioning variable for dependent missingness."""
    return "Age (bin)" if masked_var.replace(" (bin)", "") == "Debt Ratio" else "Debt Ratio (bin)"


def bin_codes(series, states):
    """0-based bin index of each row in the given ordered states (labels as strings). Raises on unseen/NaN values."""
    import pandas as pd
    c = pd.Index([str(s) for s in states]).get_indexer(series.astype(str))
    if (c < 0).any():
        raise ValueError("value not in states")
    return c


def make_mask(n_rows, level, mechanism, seed, cond_codes=None):
    if not (0.0 <= level <= 1.0):
        raise ValueError("level must be in [0, 1]")
    rng = np.random.RandomState(seed)
    u = rng.random_sample(n_rows)
    if mechanism == "mcar":
        m = u < level
        return m, float(m.mean())
    if mechanism != "dependent":
        raise ValueError("mechanism must be 'mcar' or 'dependent'")
    if cond_codes is None:
        raise ValueError("cond_codes required for 'dependent'")
    cc = np.asarray(cond_codes)
    if len(cc) != n_rows or cc.min() < 0 or cc.max() > 4:
        raise ValueError("cond_codes must be integers 0..4, one per row")
    w = cc.astype(float) + 1.0
    base = w / w.mean()
    frac = lambda s: float((u < np.minimum(1.0, s * base)).mean())
    lo, hi = 0.0, 1.0
    while frac(hi) < level and hi < 1e6:
        hi *= 2.0
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if frac(mid) < level:
            lo = mid
        else:
            hi = mid
    s = hi if abs(frac(hi) - level) <= abs(frac(lo) - level) else lo
    m = u < np.minimum(1.0, s * base)
    r = float(m.mean())
    assert abs(r - level) <= LEVELS_TOL, "realised fraction %.4f not within %.3f of %.3f" % (r, LEVELS_TOL, level)
    return m, r
'''

def _c158():
    import os, sys, hashlib, importlib.util
    import numpy as np
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    p = R + "/shared/src/masking.py"; assert not os.path.exists(p), "masking.py exists: STOP"
    ns = {}; exec(compile(MASK_SRC, "masking_candidate", "exec"), ns)
    mm = ns["make_mask"]
    rng = np.random.RandomState(1); n = 30000; cc = rng.randint(0, 5, n)
    ok = True
    def chk(name, c):
        nonlocal ok; ok &= bool(c); print("  %-62s %s" % (name, "PASS" if c else "FAIL"))
    for lev in (0.1, 0.3, 0.5):
        for mech in ("mcar", "dependent"):
            m, r = mm(n, lev, mech, 42, cc)
            chk("%s level %.1f: realised %.4f (|diff| %.4f)" % (mech, lev, r, abs(r - lev)), abs(r - lev) <= (0.005 if mech == "dependent" else 0.012) and abs(m.mean() - r) < 1e-12)
    m, r = mm(n, 0.3, "dependent", 42, cc)
    rates = [m[cc == k].mean() for k in range(5)]
    print("  dependent 0.3 masked rate by conditioning bin:", ["%.3f" % x for x in rates])
    chk("dependent: rate increases with conditioning bin", all(rates[i] < rates[i + 1] for i in range(4)))
    m2, _ = mm(n, 0.3, "dependent", 42, cc); chk("deterministic for the same seed", (m == m2).all())
    m3, _ = mm(n, 0.3, "dependent", 43, cc); chk("different seed gives a different mask", (m != m3).any())
    mc, rc = mm(n, 0.3, "mcar", 42); chk("mcar rate flat across bins (|diff| < 0.02)", max(abs(mc[cc == k].mean() - mc.mean()) for k in range(5)) < 0.02)
    me, _ = mm(n, 0.0, "mcar", 42); chk("level 0 masks nothing", not me.any())
    for bad in [dict(level=0.3, mechanism="dependent", cond_codes=None), dict(level=0.3, mechanism="xyz", cond_codes=cc),
                dict(level=0.3, mechanism="dependent", cond_codes=cc + 5), dict(level=1.5, mechanism="mcar", cond_codes=None)]:
        try: mm(n, 0.3 if False else bad["level"], bad["mechanism"], 42, bad["cond_codes"]); good = False
        except (ValueError, AssertionError): good = True
        chk("invalid input rejected: %s" % {k: (v if k != 'cond_codes' else type(v).__name__) for k, v in bad.items()}, good)
    cv = ns["cond_variable"]
    chk("cond variable: Income -> Debt Ratio (bin)", cv("Monthly Income (bin)") == "Debt Ratio (bin)")
    chk("cond variable: Debt Ratio -> Age (bin)", cv("Debt Ratio (bin)") == "Age (bin)" and cv("Debt Ratio") == "Age (bin)")
    chk("cond variable: 60-89 Days Past Due -> Debt Ratio (bin)", cv("60-89 Days Past Due (bin)") == "Debt Ratio (bin)")
    if not ok: print("\nFAILED: nothing written. STOP"); return
    with open(p, "x", encoding="utf-8") as f: f.write(MASK_SRC)
    assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(MASK_SRC.encode()).hexdigest()
    os.chmod(p, 0o444)
    spec = importlib.util.spec_from_file_location("masking_saved", p); m_ = importlib.util.module_from_spec(spec); spec.loader.exec_module(m_)
    r2 = m_.make_mask(n, 0.3, "dependent", 42, cc)[1]
    print("\nsaved shared/src/masking.py sha12 %s | re-imported from disk, realised %.4f" % (hashlib.sha256(MASK_SRC.encode()).hexdigest()[:12], r2))
_c158()

In [ ]:
# Cell 159 (06_rq3_setup.ipynb): shared/src/rq3.py (one code path, parameterised by contribution) + 0% validation. Saves only if everything passes. No restart.
RQ3_SRC = '''"""RQ3 pipeline: ONE code path parameterised by contribution ('selection' or 'monitoring').
load_context -> run_config(ctx, masked_var, level, mechanism, seed). Masks apply to the TEST set only; models are trained on complete data."""
import os
assert os.environ.get("PYTHONHASHSEED") == "0", "PYTHONHASHSEED != 0: wrong kernel"
import json, pickle, time, importlib.util
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp

R = os.environ["CREDIT_BN_ROOT"]
PROC = R + "/Data/processed/"
_L = lambda p: pickle.load(open(p, "rb"))


def _load(name, rel):
    spec = importlib.util.spec_from_file_location(name, R + "/" + rel)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m


MK = _load("masking_saved", "shared/src/masking.py")
SPEC_A = ['Age', 'Debt Ratio', 'Monthly Income', 'Open Credit Lines', 'Real Estate Loans', 'Number of Dependents']
MASK_VARS = {"selection": ['Age', 'Debt Ratio'], "monitoring": ['Credit Utilisation Rate', '30-59 Days Past Due', '90+ Days Late', '60-89 Days Past Due']}
PILOT_VAR = {"selection": "Debt Ratio", "monitoring": "60-89 Days Past Due"}
FB = ['Credit Utilisation Rate (bin)', 'Age (bin)', 'Debt Ratio (bin)', 'Open Credit Lines (bin)', '30-59 Days Past Due (bin)',
      '90+ Days Late (bin)', '60-89 Days Past Due (bin)', 'Real Estate Loans (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)']
SEL_EV = ['Age (bin)', 'Debt Ratio (bin)', 'Monthly Income (bin)', 'Number of Dependents (bin)', 'Real Estate Loans (bin)', 'Open Credit Lines (bin)']


def _states_frame(F, cols, S):
    out = {}
    for c in cols:
        s = F[c]
        if set(s.dropna().unique()) <= set(S[c]):
            out[c] = s
        else:
            m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
            assert set(m.dropna().unique()) <= set(S[c]), "label mismatch in %s" % c
            out[c] = m
    return pd.DataFrame(out)


def _xin(model, X):
    return X if hasattr(model, "feature_names_in_") else X.values


def load_context(contribution):
    assert contribution in ("selection", "monitoring")
    from pgmpy.models import DiscreteBayesianNetwork
    sp = _L(PROC + "split_indices.pkl")
    g = {"os": os, "np": np, "pd": pd, "L": _L, "PROC": PROC, "ROOT": R}
    g["train_idx"], g["test_idx"] = np.asarray(sp["train_idx"]), np.asarray(sp["test_idx"])
    g["y_train"], g["y_test"] = np.asarray(sp["y_train"]), np.asarray(sp["y_test"])
    code = "".join(json.load(open(R + "/selection/notebooks/00_precheck.ipynb"))["cells"][2]["source"])
    exec(compile(code[:code.index("# ---- validation")], "build_design_src", "exec"), g)
    raw_cols = SPEC_A if contribution == "selection" else list(g["IMPUTE_ORDER"])
    base = g["build_design"](raw_cols)
    y = g["y_test"].astype(int)
    if contribution == "selection":
        lr_gs = _L(R + "/selection/models/sel_lr_grid.pkl"); rf_gs = _L(R + "/selection/models/sel_rf_grid.pkl")
        models = {"LR": lr_gs.best_estimator_, "RF": rf_gs.best_estimator_}
        bp = _L(R + "/selection/predictions/sel_baseline_test_predictions.pkl"); bnp = _L(R + "/selection/predictions/sel_bn_test_predictions.pkl")
        p0 = {"BN": np.asarray(bnp["p"], float), "LR": np.asarray(bp["p_lr"], float), "RF": np.asarray(bp["p_rf"], float)}
        tau = {"BN": 1.0 / 11.0, "LR": float(bp["tau"]["LR"]), "RF": float(bp["tau"]["RF"])}
        b = _L(R + "/selection/models/sel_bn_final.pkl"); edges, cpds = b["model_edges"], b["cpds"]
        F = pd.read_pickle(PROC + "sel_test.pkl"); ev_cols = SEL_EV + ["M_Income", "M_Dependents"]
    else:
        models = {"LR": _L(R + "/full_variable/models/lr_final.pkl"), "RF": _L(R + "/full_variable/models/rf_final_restricted.pkl")}
        abl = _L(R + "/full_variable/predictions/ablation_predictions.pkl"); bnp = _L(R + "/full_variable/predictions/bn_test_predictions.pkl")
        p0 = {"BN": np.asarray(bnp["bn_test_probs"], float), "LR": np.asarray(abl["LR_all"], float), "RF": np.asarray(abl["RF_all"], float)}
        tau = {"BN": 0.17, "LR": 0.69, "RF": 0.67}
        d = _L(R + "/full_variable/models/bn_final_model_iteration79.pkl"); edges, cpds = d["model_edges"], d["cpds"]
        F = _L(PROC + "df_test.pkl"); ev_cols = FB + ["M_Income", "M_Dependents"]
        assert bnp["bn_test_probs"].index.equals(F.index), "BN prediction index differs from df_test: STOP"
    assert list(F.index) == list(g["test_idx"]), "test frame order differs from split: STOP"
    assert (F["Default"].to_numpy().astype(int) == y).all()
    bn = DiscreteBayesianNetwork([tuple(e) for e in edges]); bn.add_cpds(*cpds); assert bn.check_model()
    S = {n: list(bn.get_cpds(n).state_names[n]) for n in bn.nodes()}
    Fs = _states_frame(F, ev_cols, S)
    codes = np.column_stack([pd.Index(S[c]).get_indexer(Fs[c]) for c in ev_cols])
    for j, c in enumerate(ev_cols):
        assert ((codes[:, j] < 0) == Fs[c].isna().to_numpy()).all(), "unseen state in %s" % c
    return dict(name=contribution, g=g, dfc=g["dfc"], raw_cols=raw_cols, base=base, y=y, train_idx=g["train_idx"], test_idx=g["test_idx"],
                models=models, p0=p0, tau=tau, bn=bn, S=S, ev_cols=ev_cols, codes=codes, imp_cache={})


def cond_codes(ctx, masked_raw, split):
    cv = MK.cond_variable(masked_raw).replace(" (bin)", "")
    idx = ctx["test_idx"] if split == "test" else ctx["train_idx"]
    s = ctx["g"]["cut_edges"](ctx["dfc"].loc[idx, cv], ctx["g"]["EDGES"][cv])
    c = np.asarray(s.cat.codes).astype(int)
    assert c.min() >= 0 and c.max() <= 4, "conditioning codes out of range"
    return c


def bn_posteriors(ctx, masked_raw, mask, check_bp=True):
    from pgmpy.inference import VariableElimination, BeliefPropagation
    bn, S, ev_cols = ctx["bn"], ctx["S"], ctx["ev_cols"]
    c2 = ctx["codes"].copy()
    if masked_raw is not None:
        c2[mask, ev_cols.index(masked_raw + " (bin)")] = -1
    u, inv = np.unique(c2, axis=0, return_inverse=True); inv = np.asarray(inv).ravel()
    ev_of = lambda r: {c: S[c][i] for c, i in zip(ev_cols, r) if i >= 0}
    eng = VariableElimination(bn); pu = np.full(len(u), np.nan); t0 = time.time()
    for k, r in enumerate(u):
        try:
            with np.errstate(invalid="ignore", divide="ignore"):
                p = float(eng.query(variables=["Default"], evidence=ev_of(r), show_progress=False).get_value(Default=1))
        except Exception:
            p = np.nan
        pu[k] = p if (np.isfinite(p) and 0 <= p <= 1) else np.nan
    info = dict(n_patterns=int(len(u)), t_ve=time.time() - t0, bp_maxdiff=None, bp_n=0, t_bp=0.0)
    if check_bp:
        valid = np.where(np.isfinite(pu))[0]
        pick = valid if len(valid) <= 500 else np.random.RandomState(42).choice(valid, 500, replace=False)
        bpe = BeliefPropagation(bn); t0 = time.time(); dmax = 0.0
        for k in pick:
            pb = float(bpe.query(variables=["Default"], evidence=ev_of(u[k]), show_progress=False).get_value(Default=1))
            dmax = max(dmax, abs(pb - pu[k]))
        info.update(bp_maxdiff=dmax, bp_n=int(len(pick)), t_bp=time.time() - t0)
    return pu[inv], info


def _imp_cols(ctx):
    return [c for c in ctx["g"]["IMPUTE_ORDER"] if c in ctx["raw_cols"]]


def get_refit_imputer(ctx, masked_raw, level, mechanism):
    key = (masked_raw, float(level), mechanism)
    if key in ctx["imp_cache"]:
        imp, t, real = ctx["imp_cache"][key]; return imp, t, real, True
    imp_cols = _imp_cols(ctx)
    tr = ctx["dfc"].loc[ctx["train_idx"], imp_cols].astype(float)
    if "Monthly Income" in imp_cols:
        tr["Monthly Income"] = np.log1p(tr["Monthly Income"])
    cc = None if mechanism == "mcar" else cond_codes(ctx, masked_raw, "train")
    tm, real = MK.make_mask(len(tr), level, mechanism, 42, cc)
    tr.loc[tm, masked_raw] = np.nan
    t0 = time.time(); imp = IterativeImputer(random_state=42).fit(tr); t = time.time() - t0
    ctx["imp_cache"][key] = (imp, t, real)
    return imp, t, real, False


def masked_matrix(ctx, masked_raw, test_mask, mode, imp=None):
    g, base = ctx["g"], ctx["base"]
    imp_cols = _imp_cols(ctx); des_cols = [c for c in g["DESIGN_ORDER"] if c in ctx["raw_cols"]]
    tr = ctx["dfc"].loc[ctx["train_idx"], imp_cols].astype(float); te = ctx["dfc"].loc[ctx["test_idx"], imp_cols].astype(float)
    if "Monthly Income" in imp_cols:
        tr["Monthly Income"] = np.log1p(tr["Monthly Income"]); te["Monthly Income"] = np.log1p(te["Monthly Income"])
    if mode == "refit":
        te.loc[test_mask, masked_raw] = np.nan; use = imp
    elif mode in ("median", "mean"):
        te.loc[test_mask, masked_raw] = tr[masked_raw].median() if mode == "median" else tr[masked_raw].mean(); use = base["imputer"]
    else:
        raise ValueError(mode)
    te_i = pd.DataFrame(use.transform(te), columns=imp_cols, index=te.index)
    if "Monthly Income" in imp_cols:
        te_i["Monthly Income"] = np.expm1(te_i["Monthly Income"]).clip(lower=0)
    s_te = pd.DataFrame(index=te_i.index)
    for c in des_cols:
        s_te[c + " (bin)"] = g["cut_edges"](te_i[c], g["EDGES"][c]) if c in g["EDGE_VARS"] else g["bin_natural_category"](te_i[c], 3)
    s_te = s_te.astype(str)
    return pd.DataFrame(base["encoder"].transform(s_te), columns=base["Xte"].columns, index=te_i.index)


def _row(ctx, model, imp, p, valid):
    y = ctx["y"][valid]; pv = p[valid]; p0v = ctx["p0"][model][valid]; t = ctx["tau"][model]
    a, a0 = roc_auc_score(y, pv), roc_auc_score(y, p0v); d = np.abs(pv - p0v)
    return dict(model=model, imputation=imp, auc=a, gini=2 * a - 1, ks=float(ks_2samp(pv[y == 1], pv[y == 0]).statistic),
                ap=float(average_precision_score(y, pv)), auc_0pct_same_rows=a0, delta_auc=a - a0, mean_abs_dp=float(d.mean()),
                share_dp_gt_0_01=float((d > 0.01).mean()), flip_share=float(((pv >= t) != (p0v >= t)).mean()), tau=t, n_valid=int(valid.sum()))


def run_config(ctx, masked_raw, level, mechanism, seed, secondary="median", bp_check=True, stop_frac=0.001, return_preds=False):
    y = ctx["y"]; n = len(y); p0 = ctx["p0"]; models = ctx["models"]; T = {}
    cc = None if mechanism == "mcar" else cond_codes(ctx, masked_raw, "test")
    mask, realised = MK.make_mask(n, level, mechanism, seed, cc)
    pbn, info = bn_posteriors(ctx, masked_raw, mask, check_bp=bp_check)
    T["bn_ve"], T["bn_bp"] = info["t_ve"], info["t_bp"]
    bad0, bad1 = ~np.isfinite(p0["BN"]), ~np.isfinite(pbn); valid = ~(bad0 | bad1)
    excl = dict(baseline=int(bad0.sum()), new_in_config=int((bad1 & ~bad0).sum()), total=int((~valid).sum()))
    if excl["total"] / n > stop_frac:
        raise RuntimeError("STOP: %d rows (%.3f%%) excluded in %s %s %.2f %s seed %d (limit %.1f%%)" % (excl["total"], 100.0 * excl["total"] / n, ctx["name"], masked_raw, level, mechanism, seed, 100 * stop_frac))
    imp, t_fit, _, cached = get_refit_imputer(ctx, masked_raw, level, mechanism)
    T["imputer_refit"] = 0.0 if cached else t_fit; T["imputer_cached"] = cached
    preds = {"BN": pbn}; rows = [_row(ctx, "BN", "evidence removed", pbn, valid)]
    for lab, mode in (("primary", "refit"), (secondary, secondary)):
        t0 = time.time(); X = masked_matrix(ctx, masked_raw, mask, mode, imp=imp); T[lab + "_transform"] = time.time() - t0
        for k in ("LR", "RF"):
            t0 = time.time(); p = models[k].predict_proba(_xin(models[k], X))[:, 1]; T[k.lower() + "_" + lab] = time.time() - t0
            preds[k + "_" + lab] = p
            rows.append(_row(ctx, k, "refit IterativeImputer (primary)" if lab == "primary" else "train %s (secondary)" % secondary, p, valid))
    out = dict(contribution=ctx["name"], masked_var=masked_raw, level=level, mechanism=mechanism, seed=seed, realised=realised,
               n_patterns=info["n_patterns"], bp_maxdiff=info["bp_maxdiff"], bp_n=info["bp_n"], excluded=excl, timing=T, rows=rows)
    if return_preds:
        out.update(preds=preds, mask=mask, valid=valid)
    return out


def validate_0pct(ctx, expected):
    y, p0, models, base = ctx["y"], ctx["p0"], ctx["models"], ctx["base"]; out = {}
    if ctx["name"] == "monitoring":
        Xl = _L(PROC + "X_test_df.pkl")
        assert np.array_equal(base["Xte"].values, Xl.values) and list(base["Xte"].columns) == list(Xl.columns), "rebuilt design != locked X_test_df: STOP"
        print("  rebuilt monitoring design equals the locked X_test_df exactly")
    for k in ("LR", "RF"):
        p = models[k].predict_proba(_xin(models[k], base["Xte"]))[:, 1]
        d = float(np.max(np.abs(p - p0[k]))); a = roc_auc_score(y, p)
        print("  %s object on the rebuilt design: AUC %.4f (expected %.4f) | max |p - saved| %.3e" % (k, a, expected[k], d))
        assert d <= 1e-12 and round(a, 4) == expected[k], "%s does not reproduce the saved predictions: STOP" % k
        out[k] = dict(auc=a, max_abs_diff=d)
    r = run_config(ctx, MASK_VARS[ctx["name"]][0], 0.0, "mcar", 42, return_preds=True)
    pb = r["preds"]["BN"]
    assert np.array_equal(np.isfinite(pb), np.isfinite(p0["BN"])), "BN NaN positions differ from saved: STOP"
    ok = np.isfinite(pb); d = float(np.max(np.abs(pb[ok] - p0["BN"][ok]))); a = roc_auc_score(y[ok], pb[ok])
    print("  BN at 0%%: %d rows valid | AUC %.4f (expected %.4f) | max |p - saved| %.3e | unique patterns %d | BP check on %d patterns max diff %.3e" % (
        ok.sum(), a, expected["BN"], d, r["n_patterns"], r["bp_n"], r["bp_maxdiff"]))
    assert d <= 1e-10 and round(a, 4) == expected["BN"] and int(ok.sum()) == expected["n_bn"] and r["bp_maxdiff"] <= 1e-10, "BN validation failed: STOP"
    out["BN"] = dict(auc=a, max_abs_diff=d, n_valid=int(ok.sum()), n_patterns=r["n_patterns"], bp_maxdiff=r["bp_maxdiff"], bp_n=r["bp_n"])
    for k in ("LR", "RF"):
        for lab in ("primary", "median"):
            dd = float(np.max(np.abs(r["preds"][k + "_" + lab] - p0[k])))
            print("  level-0 masked pipeline, %s %s: max |p - saved| %.3e" % (k, lab, dd))
            assert dd <= 1e-9, "level-0 pipeline does not reproduce baseline: STOP"
            out[k + "_level0_" + lab] = dd
    assert all(abs(x["delta_auc"]) < 1e-9 and x["flip_share"] == 0 for x in r["rows"]), "level-0 delta/flip not zero: STOP"
    print("  level 0: delta AUC 0 and flip share 0 for every row of the table; excluded rows: %s" % r["excluded"])
    out["excluded_at_0pct"] = r["excluded"]; out["tau"] = ctx["tau"]
    return out
'''

def _c159():
    import os, sys, json, time, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    OUTJ = {"selection": "selection/tables/sel_rq3_validation_0pct.json", "monitoring": "monitoring_rq3/mon_rq3_validation_0pct.json"}
    for rel in OUTJ.values(): ss.check_path(os.path.join(R, rel))
    mp = R + "/shared/src/rq3.py"; assert not os.path.exists(mp), "rq3.py exists: STOP"
    ns = {}; exec(compile(RQ3_SRC, "rq3_candidate", "exec"), ns)
    EXPECT = {"selection": dict(BN=0.6627, LR=0.6904, RF=0.6926, n_bn=30000), "monitoring": dict(BN=0.8519, LR=0.8618, RF=0.8603, n_bn=29998)}
    res = {}
    for c in ("selection", "monitoring"):
        print("=== %s ===" % c); t0 = time.time()
        ctx = ns["load_context"](c); print("  context loaded in %.0fs (BN %d nodes, design %s)" % (time.time() - t0, len(ctx["bn"].nodes()), ctx["base"]["Xte"].shape))
        t0 = time.time(); res[c] = ns["validate_0pct"](ctx, EXPECT[c]); print("  validated in %.0fs\n" % (time.time() - t0))
    with open(mp, "x", encoding="utf-8") as f: f.write(RQ3_SRC)
    h = hashlib.sha256(open(mp, "rb").read()).hexdigest(); assert h == hashlib.sha256(RQ3_SRC.encode()).hexdigest()
    os.chmod(mp, 0o444)
    for c in res:
        data = json.dumps(dict(contribution=c, rq3_sha12=h[:12], validation=res[c], note="0% masking validation; all asserts passed"), indent=1, default=float).encode()
        p = ss.check_path(os.path.join(R, OUTJ[c]))
        with open(p, "xb") as f: f.write(data)
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
        print("saved %s sha12 %s" % (OUTJ[c], hashlib.sha256(data).hexdigest()[:12]))
    print("saved shared/src/rq3.py sha12 %s\nALL PASS" % h[:12])
_c159()

In [ ]:
# Cell 160 (06_rq3_setup.ipynb): Part D pilot, TIMING ONLY. Run after Cell 159 printed ALL PASS. Saves two timing files (exclusive create). No restart.
def _c160():
    import os, sys, json, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, R + "/shared/src"); import safe_save as ss
    OUTJ = {"selection": "selection/tables/sel_rq3_pilot_timing.json", "monitoring": "monitoring_rq3/mon_rq3_pilot_timing.json"}
    for rel in OUTJ.values(): ss.check_path(os.path.join(R, rel))
    spec = importlib.util.spec_from_file_location("rq3", R + "/shared/src/rq3.py"); rq3 = importlib.util.module_from_spec(spec); spec.loader.exec_module(rq3)
    NV = {"selection": 2, "monitoring": 4}; LEVELS, MECHS = 3, 2
    saved = {}
    for c in ("selection", "monitoring"):
        var = rq3.PILOT_VAR[c]; ctx = rq3.load_context(c)
        r = rq3.run_config(ctx, var, 0.50, "dependent", 42, bp_check=True)
        T = r["timing"]
        print("=== %s pilot: %s, level 0.50, dependent, seed 42 ===" % (c, var))
        print("  realised masked fraction %.4f | excluded rows %s" % (r["realised"], r["excluded"]))
        print("  BN: %d unique evidence patterns | VE %.1fs | BP check on %d patterns, max diff %.2e, %.1fs" % (r["n_patterns"], T["bn_ve"], r["bp_n"], r["bp_maxdiff"], T["bn_bp"]))
        print("  imputer refit (IterativeImputer, train with mask): %.1fs" % T["imputer_refit"])
        print("  primary: transform %.1fs | LR predict %.2fs | RF predict %.2fs" % (T["primary_transform"], T["lr_primary"], T["rf_primary"]))
        print("  secondary (median): transform %.1fs | LR predict %.2fs | RF predict %.2fs" % (T["median_transform"], T["lr_median"], T["rf_median"]))
        per_seed = T["bn_ve"] + T["bn_bp"] + sum(T[k] for k in ("primary_transform", "lr_primary", "rf_primary", "median_transform", "lr_median", "rf_median"))
        per_seed_nobp = per_seed - T["bn_bp"]; once = T["imputer_refit"]; combos = NV[c] * LEVELS * MECHS
        ex = {}
        for seeds in (10, 5):
            tot = combos * (once + seeds * per_seed); tot2 = combos * (once + seeds * per_seed_nobp)
            ex[seeds] = dict(runs=combos * seeds, minutes=tot / 60, minutes_without_bp_check=tot2 / 60)
            print("  FULL GRID, %2d seeds: %d variable-level-mechanism combos x %d seeds = %d runs | about %.0f min (%.1f h) | %.0f min if the BP check is skipped after the first seed" % (
                seeds, combos, seeds, combos * seeds, tot / 60, tot / 3600, tot2 / 60))
        print("  per seed-run: %.1fs (BN VE %.1fs, BP %.1fs, LR/RF + transforms %.1fs); imputer refit once per combo: %.1fs\n" % (per_seed, T["bn_ve"], T["bn_bp"], per_seed - T["bn_ve"] - T["bn_bp"], once))
        saved[c] = dict(variable=var, level=0.5, mechanism="dependent", seed=42, n_patterns=r["n_patterns"], timing=T, excluded=r["excluded"], bp_maxdiff=r["bp_maxdiff"],
                        extrapolation=ex, assumptions="levels 0.10/0.30/0.50; 2 mechanisms; serial; pilot timing at level 0.50 applied to all levels; no results kept")
    for c, d in saved.items():
        data = json.dumps(d, indent=1, default=float).encode(); p = ss.check_path(os.path.join(R, OUTJ[c]))
        with open(p, "xb") as f: f.write(data)
        print("saved %s sha12 %s" % (OUTJ[c], hashlib.sha256(data).hexdigest()[:12]))
_c160()

In [ ]:
# Cell 161 (06_rq3_setup.ipynb): Phase 6a manifest/git DRY RUN. Writes NOTHING. No restart.
def _c161():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    def git(*a):
        return subprocess.run(["git", "-C", R, *a], capture_output=True, text=True).stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    print("DRY_RUN = True")
    print("manifest rows now:", len(existing), "(expected 176) | ends with newline:", raw.endswith(b'\n'))
    tag = git("rev-list", "-n", "1", "full-variable-final")
    print("tag points to:", tag[:12], "| OK:", tag.startswith(TAG12))
    print("git status before:", git("status", "--porcelain").splitlines())
    rows = []
    for top in ("selection", "shared", "monitoring_rq3"):
        for dp, dn, fn in os.walk(os.path.join(R, top)):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in sorted(fn):
                if f.endswith(".pyc") or f == ".DS_Store":
                    continue
                p = os.path.join(dp, f); rel = os.path.relpath(p, R)
                if rel not in existing:
                    rows.append((rel, os.path.getsize(p), sha(p)[:12], time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))))
    print("\nrows to append (%d):" % len(rows))
    for r in sorted(rows): print("  ", *r)
    print("expected manifest rows after:", len(existing) + len(rows))
    print("\nshared/src/safe_save.py is already in the manifest with its OLD hash; it was modified in this phase.")
    print("would stage ONLY: MANIFEST.csv, selection/notebooks/06_rq3_setup.ipynb, shared/src/checks.py, masking.py, rq3.py, safe_save.py")
    print("(not 03_/04_ notebooks, Reminder.md, Writing_Edits.md)")
    print("DRY RUN ONLY: nothing written.")
_c161()

In [ ]:
# Cell 162 (06_rq3_setup.ipynb): REAL RUN (approved). Manifest append (13 rows) + NEW git commit. Tag never touched.
def _c162():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    NOTE = "added after Phase 6a, 2026-10-15"
    NB = "selection/notebooks/06_rq3_setup.ipynb"
    F = [("monitoring_rq3/mon_rq3_pilot_timing.json", "c480b84eac35", "monitoring_rq3"),
         ("monitoring_rq3/mon_rq3_validation_0pct.json", "10c25f30e051", "monitoring_rq3"),
         (NB, None, "selection"),
         ("selection/tables/sel_ablationLR_convergence_check.csv", "1b8507bfe8b7", "selection"),
         ("selection/tables/sel_ablationLR_convergence_check.json", "7bf6cb936466", "selection"),
         ("selection/tables/sel_rq3_pilot_timing.json", "d48090221231", "selection"),
         ("selection/tables/sel_rq3_validation_0pct.json", "cfe24f57dc5e", "selection"),
         ("selection/tables/sel_superseded_notes.json", "4cbc287d551e", "selection"),
         ("shared/src/checks.py", "ef542907c037", "shared"),
         ("shared/src/masking.py", "c6094c2b232e", "shared"),
         ("shared/src/rq3.py", "5647309a230d", "shared"),
         ("shared/src/safe_save_before_mon.py.bak", "d3a3641043fe", "shared")]
    SS = "shared/src/safe_save.py"
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0:
            raise RuntimeError(f"git {' '.join(a)} failed: {r.stderr.strip()[:300]}")
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw0 = open(mp, 'rb').read()
    recs = list(csv.DictReader(open(mp, newline='')))
    existing = {r['path'] for r in recs}
    assert raw0.endswith(b'\n') and len(recs) == 176, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12), "tag moved: STOP"
    ss_new = sha(os.path.join(R, SS))
    assert ss_new.startswith("afff88e2b30a"), "safe_save.py hash is not the validated new version: STOP"
    assert not any(r['path'] == SS and r.get('sha256', '').startswith("afff88e2b30a") for r in recs), "new safe_save row already present: STOP"
    rows = []
    for rel, h12, role in F:
        p = os.path.join(R, rel)
        assert rel not in existing and os.path.exists(p), f"unexpected state for {rel}: STOP"
        h = sha(p)
        if h12 is not None:
            assert h.startswith(h12), f"hash changed since dry run for {rel}: STOP"
        else:
            print("notebook hash recorded now: %s (dry run showed 2194d76a0fc6)" % h[:12])
        rows.append([rel, os.path.getsize(p), h, role, time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), NOTE])
    p = os.path.join(R, SS)
    rows.append([SS, os.path.getsize(p), ss_new, "shared", time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))),
                 NOTE + "; MODIFIED: adds mon_ prefix inside monitoring_rq3/ only; supersedes the earlier safe_save.py row (old version kept as safe_save_before_mon.py.bak)"])
    with open(mp, 'a', newline='') as fh:
        csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=''))))
    assert n == 189 and open(mp, 'rb').read().startswith(raw0), "manifest check failed: STOP (not committed)"
    git("add", "MANIFEST.csv", NB, "shared/src/checks.py", "shared/src/masking.py", "shared/src/rq3.py", SS)
    ident = []
    if not git("config", "user.name", check=False) or not git("config", "user.email", check=False):
        ident = ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 6a: RQ3 setup (checks, masking, rq3 modules, safe_save mon_ extension), validation and pilot timing")
    print("manifest rows now:", n, "(expected 189)")
    print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12))
    print("git status:", git("status", "--porcelain").splitlines())
_c162()